# 01 Data check (B-03)

Verifies MIT1003, fixes the human-map blur (D-05), and checks fixation scaling (ARCHITECTURE §7, checks 4 and 5).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from humper import data, maps

OUT = data.REPO_ROOT / "results"
FIG = OUT / "figures" / "data_check"
TAB = OUT / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)
INK = "#2a6fb0"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

df = data.load_images()
assert len(df) == 1003 and df.image_id.is_unique
summary = pd.Series({
    "images": len(df),
    "distinct_sizes": df[["width", "height"]].drop_duplicates().shape[0],
    "max_side": int(df[["width", "height"]].max(axis=1).max()),
    "min_long_side": int(df[["width", "height"]].max(axis=1).min()),
    "landscape_share": round(float((df.width > df.height).mean()), 3),
    "fixations_min": int(df.n_fixations.min()),
    "fixations_median": float(df.n_fixations.median()),
    "fixations_max": int(df.n_fixations.max()),
})
summary.to_csv(TAB / "data_summary.csv", header=["value"])
summary

In [ ]:
counts = df.n_fixations.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(counts.index, counts.values, width=0.85, color=INK)
ax.set(title="Unique fixation points per image (MIT1003)", xlabel="Fixation points", ylabel="Images")
ax.grid(axis="y", color="#e5e5e5", linewidth=0.8); ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig(FIG / "fixation_counts.png", dpi=160); plt.show()

## Blur fit

The dataset documents no blur size or viewing geometry, so we recover σ by matching its own maps.

In [ ]:
SIGMAS = [15, 20, 22, 23, 24, 25, 26, 27, 28, 30, 35]

def cc_to_dataset(row, sigma):
    ours = maps.human_map_native(data.load_fixations(row.image_id), (row.width, row.height), sigma)
    return np.corrcoef(ours.ravel(), data.load_fix_map(row.image_id).ravel())[0, 1]

sample = df.sample(100, random_state=0)
fit = pd.DataFrame([
    {"sigma_px": s, "mean_cc": np.mean([cc_to_dataset(r, s) for r in sample.itertuples()])}
    for s in SIGMAS
])
fit.to_csv(TAB / "sigma_fit.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(fit.sigma_px, fit.mean_cc, color=INK, linewidth=2, marker="o", markersize=7)
best = fit.loc[fit.mean_cc.idxmax()]
ax.annotate(f"best: σ = {best.sigma_px:.0f} px", (best.sigma_px, best.mean_cc), xytext=(0, -22),
            textcoords="offset points", ha="center")
ax.set(title="Our blurred map vs the dataset map (100 images)", xlabel="Gaussian σ (native pixels)", ylabel="Mean CC")
ax.grid(axis="y", color="#e5e5e5", linewidth=0.8); ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig(FIG / "sigma_fit.png", dpi=160); plt.show()
assert best.sigma_px == maps.HUMAN_SIGMA_PX
fit

In [ ]:
all_cc = pd.Series([cc_to_dataset(r, maps.HUMAN_SIGMA_PX) for r in df.itertuples()], index=df.image_id)
assert all_cc.min() > 0.99, all_cc.idxmin()
stats = all_cc.describe(); stats.to_csv(TAB / "human_map_vs_dataset.csv", header=["cc"]); stats

## Overlays

The figure contains dataset images, so it is gitignored.

In [ ]:
rows = df.sample(5, random_state=1).itertuples()
fig, axes = plt.subplots(5, 4, figsize=(11, 12))
for ax_row, r in zip(axes, rows):
    image = data.load_image(r.image_id)
    xy = data.load_fixations(r.image_id)
    canvas_xy = maps.scale_fixations(xy, (r.width, r.height))
    ax_row[0].imshow(image); ax_row[0].scatter(*xy.T, s=14, c="#ffd21f", edgecolors="black", linewidths=0.4)
    ax_row[1].imshow(maps.human_map_native(xy, (r.width, r.height)), cmap="magma", vmin=0, vmax=1)
    ax_row[2].imshow(data.load_fix_map(r.image_id), cmap="magma", vmin=0, vmax=1)
    ax_row[3].imshow(image.resize((maps.CANVAS_SIZE, maps.CANVAS_SIZE)))
    ax_row[3].scatter(*canvas_xy.T, s=14, c="#ffd21f", edgecolors="black", linewidths=0.4)
    for ax in ax_row: ax.axis("off")
for ax, title in zip(axes[0], ["Image + fixations", "Our map (σ = 24 px)", "Dataset map", "224×224 + scaled fixations"]):
    ax.set_title(title)
fig.tight_layout(); fig.savefig(FIG / "fixation_overlays.png", dpi=130); plt.show()